# Tranzia API Integration Tutorial

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/afzal-xyz/tranzia-receipt-verifier/blob/main/examples/api_integration_tutorial.ipynb)

This notebook demonstrates the complete lifecycle of integrating with the Tranzia Safety API.

**Lifecycle:**
1. **Onboarding**: Create an account and generate an API key programmatically.
2. **Scoring**: Get raw safety scores for routes.
3. **Decision**: Generate a defensible "Decision Receipt" (The Core Product).
4. **Verification**: Verify the receipt's integrity.

**Base URL**: `https://tranzia.com`

In [ ]:
import requests
import json
import uuid
from datetime import datetime, timedelta

BASE_URL = "https://tranzia.com"

# Optional: If you already have a key, set it here. Otherwise we generate one.
API_KEY = None

## 1. Onboarding (Programmatic)
Most users sign up via the Dashboard, but you can also do it via API.

In [ ]:
if not API_KEY:
    # 1. Create Account
    email = f"demo_{uuid.uuid4().hex[:8]}@example.com"
    print(f"Creating account for {email}...")
    
    signup_resp = requests.post(f"{BASE_URL}/v1/accounts/signup", json={
        "name": "Demo User",
        "email": email,
        "password": "DemoPass123!"
    })
    signup_resp.raise_for_status()
    account_id = signup_resp.json()["id"]
    print(f"Account Created: {account_id}")

    # 2. Generate API Key
    # (In production, you would authenticate first. For V1 demo, we allow key gen via account_id)
    key_resp = requests.post(f"{BASE_URL}/v1/accounts/keys", 
                             params={"account_id": account_id}, 
                             json={"name": "Tutorial Key"})
    key_resp.raise_for_status()
    
    API_KEY = key_resp.json()["key"]
    print(f" API Key Generated: {API_KEY}")
else:
    print(f"Using Existing API Key: {API_KEY}")

## 2. Route Scoring (Raw Data)
Use `POST /v1/score` when you just want the risk numbers (0-10) and factors for a specific route.

In [ ]:
# Ensure departure time is in the future for valid transit schedules
future_time = (datetime.utcnow() + timedelta(hours=1)).strftime("%Y-%m-%dT%H:%M:%SZ")

score_payload = {
    "routes": [
        {
            "city_id": "nyc",
            "origin": "Columbus Circle, NY",
            "destination": "Times Square, NY",
            "departure_time": future_time
        }
    ]
}

headers = {
    "X-API-Key": API_KEY,
    "Content-Type": "application/json"
}

resp = requests.post(f"{BASE_URL}/v1/score", json=score_payload, headers=headers)
print(json.dumps(resp.json(), indent=2))

## 3. Decision Receipt (The Core Product)
Use `POST /v1/decision` when you need a **defensible audit trail**. 
This endpoint returns a signed/hashed "Receipt" that includes the context, the advice given, and proof of integrity.
**This is the endpoint you use for Duty-of-Care compliance.**

In [ ]:
receipt_payload = {
    "context": {
        "local_time": future_time, # Use same dynamic time
        "mode": "walk",
        "day_of_week": datetime.utcnow().strftime("%A")
    },
    "route": {
        "origin": {"lat": 40.7681, "lng": -73.9819}, # Columbus Circle
        "destination": {"lat": 40.7580, "lng": -73.9855} # Times Square
    },
    "policy_ref": {
        "policy_id": "corp-travel-night"
    }
}

resp = requests.post(f"{BASE_URL}/v1/decision", json=receipt_payload, headers=headers)
receipt = resp.json()

print(f"Receipt ID: {receipt['receipt_id']}")
print(f"Integrity Hash: {receipt['integrity']['canonical_receipt_hash']}")
print(f"Options Generated: {len(receipt['options'])}")

# Save for verification
with open("my_receipt.json", "w") as f:
    json.dump(receipt, f, indent=2)

## 4. Retrieve & Verify
You can retrieve a receipt later using its ID.

In [ ]:
receipt_id = receipt['receipt_id']
audit_resp = requests.get(f"{BASE_URL}/v1/audit/{receipt_id}", headers=headers)
audit_data = audit_resp.json()

# Check the HTTP headers for provenance
etag = audit_resp.headers.get("ETag")
print(f"Server ETag: {etag}")
print(f"Server Hash: {audit_resp.headers.get('X-Receipt-Hash')}")